# SARIMAX Model

## Objective

This notebook develops a Seasonal Autoregressive Integrated Moving Average with Exogenous Variables (SARIMAX) model for forecasting weekly German electricity demand.

Calendar-based explanatory variables are incorporated into the model to capture temporal patterns beyond those represented by the seasonal autoregressive structure. Model performance is evaluated using RMSE, MAE and MAPE and compared with the benchmark and SARIMA models.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error

from statsmodels.tsa.statespace.sarimax import SARIMAX

from statsmodels.stats.diagnostic import acorr_ljungbox

from statsmodels.graphics.gofplots import qqplot

In [2]:
weekly_load = pd.read_csv(
    "weekly_load.csv",
    index_col=0,
    parse_dates=True
)

weekly_load.head()

,load
timestamp,
2015-01-04 00:00:00+00:00,47233.739583
2015-01-11 00:00:00+00:00,56191.101190
2015-01-18 00:00:00+00:00,57672.678571
2015-01-25 00:00:00+00:00,58613.303571
2015-02-01 00:00:00+00:00,58734.029762


In [3]:
forecast_horizon = 104

train = weekly_load.iloc[:-forecast_horizon]

test = weekly_load.iloc[-forecast_horizon:]

print("Training observations:", len(train))
print("Testing observations:", len(test))

Training observations: 197
Testing observations: 104


## Feature Engineering

Calendar-based explanatory variables are generated from the timestamp to provide additional information to the SARIMAX model. Both categorical calendar features and cyclical transformations are included to better represent recurring seasonal behaviour.

In [4]:
weekly_load["month"] = weekly_load.index.month

weekly_load["quarter"] = weekly_load.index.quarter

weekly_load["week"] = weekly_load.index.isocalendar().week.astype(int)

weekly_load["year"] = weekly_load.index.year

In [5]:
weekly_load["month_sin"] = np.sin(2 * np.pi * weekly_load["month"] / 12)

weekly_load["month_cos"] = np.cos(2 * np.pi * weekly_load["month"] / 12)

weekly_load["week_sin"] = np.sin(2 * np.pi * weekly_load["week"] / 52)

weekly_load["week_cos"] = np.cos(2 * np.pi * weekly_load["week"] / 52)

In [6]:
train = weekly_load.iloc[:-forecast_horizon]

test = weekly_load.iloc[-forecast_horizon:]

feature_columns = [
    "month",
    "quarter",
    "week",
    "year",
    "month_sin",
    "month_cos",
    "week_sin",
    "week_cos"
]

exog_train = train[feature_columns]

exog_test = test[feature_columns]

## SARIMAX Model Fitting

A SARIMAX model is fitted using the optimal SARIMA parameters identified previously. Calendar-based explanatory variables are included as exogenous predictors to improve forecasting performance.

In [7]:
sarimax_model = SARIMAX(
    train["load"],
    exog=exog_train,
    order=(1, 1, 2),
    seasonal_order=(0, 1, 1, 52),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarimax_results = sarimax_model.fit(disp=False)

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [8]:
print(sarimax_results.summary())

                                      SARIMAX Results                                       
Dep. Variable:                                 load   No. Observations:                  197
Model:             SARIMAX(1, 1, 2)x(0, 1, [1], 52)   Log Likelihood                -778.063
Date:                              Thu, 02 Jul 2026   AIC                           1582.127
Time:                                      10:35:13   BIC                           1614.479
Sample:                                  01-04-2015   HQIC                          1595.167
                                       - 10-07-2018                                         
Covariance Type:                                opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
month         34.0699   1.08e+04      0.003      0.997   -2.11e+04    2.12e+04
quarter     1361.03

In [9]:
with open("sarimax_model_summary.txt", "w") as f:
    f.write(sarimax_results.summary().as_text())